# 04 Homework 04 ETM — Node Classification

**Task 5 of the Final Assignment**

This notebook builds a **CellComplex** from the Brownstone floor plan room volumes, assigns
room-type labels and door-type apertures, exports the graph to CSV in the MSD model schema,
then runs the pretrained `msd_node_classifier.pt` to predict each room's type.

## Pipeline
1. Load room OBJs → build CellComplex with room-type labels  
2. Load door OBJs → add as apertures  
3. `Graph.ByTopology(cc, directApertures=True)` → adjacency graph through doors  
4. Compute zoning and connectivity one-hot features per node/edge  
5. Export to CSV (MSD schema)  
6. Load with `PyG.ByCSVPath`, load pretrained model, predict  
7. Visualise true vs predicted labels

## MSD label → room-type mapping
| `room_type` | `label` | Zoning class |
|---|---:|---|
| `bedroom` | `0` | Private / static |
| `livingroom` | `1` | Living / dynamic |
| `kitchen` | `2` | Living / dynamic |
| `dining` | `3` | Living / dynamic |
| `corridor` | `4` | Living / dynamic |
| `stairs` | `5` | Service / functional |
| `storeroom` | `6` | Service / functional |
| `bathroom` | `7` | Service / functional |
| `balcony` | `8` | Outdoor / semi-outdoor |

**Room OBJs:** `Homework04/Objects/*.obj`  
**Apertures:** `doors2.obj` (door) · `Entrance door.obj` (entrance_door) · `Passage Door.objbak` (passage)  
**Reference:** instructor @channel pattern + HW02 `Cell.ByFaces` approach

## 1. Imports

In [77]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color
from topologicpy.PyG import PyG
import pandas as pd
import numpy as np
import os
from collections import Counter

## 2. Version check

In [78]:
print("This notebook requires topologicpy 0.9.43 or newer.")
print(Helper.Version())

This notebook requires topologicpy 0.9.43 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Renderer

In [79]:
renderer = "vscode"

## 4. Room-type and door-type mappings

### Zoning classes (for node features)
| Zoning | One-hot index | Room types |
|---|---:|---|
| Private/static | `0` | bedroom |
| Living/dynamic | `1` | livingroom, kitchen, dining, corridor |
| Service/functional | `2` | stairs, storeroom, bathroom |
| Outdoor/semi-outdoor | `3` | balcony |

### Connectivity classes (for node and edge features)
| Connection type | One-hot index | Meaning |
|---|---:|---|
| passage | `0` | Open opening / corridor connection |
| door | `1` | Standard interior door |
| entrance_door | `2` | Exterior / entrance door |

## 4. MSD label and feature mappings

In [80]:
ROOM_LABEL = {
    "bedroom": 0, "livingroom": 1, "kitchen": 2, "dining": 3,
    "corridor": 4, "stairs": 5, "storeroom": 6, "bathroom": 7, "balcony": 8,
}
ZONING = {
    "bedroom":   [1,0,0,0], "livingroom": [0,1,0,0], "kitchen":   [0,1,0,0],
    "dining":    [0,1,0,0], "corridor":   [0,1,0,0], "stairs":    [0,0,1,0],
    "storeroom": [0,0,1,0], "bathroom":   [0,0,1,0], "balcony":   [0,0,0,1],
}
NODE_CONNECTIVITY = {
    "bedroom":   [0,1,0], "livingroom": [0,1,0], "kitchen":  [0,1,0],
    "dining":    [0,1,0], "corridor":   [1,0,0], "stairs":   [1,0,0],
    "storeroom": [0,1,0], "bathroom":   [0,1,0], "balcony":  [0,1,0],
}
DOOR_CONNECTIVITY = {
    "passage":       [1,0,0],
    "door":          [0,1,0],
    "entrance_door": [0,0,1],
}

# Colors matching 02_Homework04_etm.ipynb exactly
ROOM_COLOR = {
    "bedroom":    "#E63946",
    "bathroom":   "#4CC9F0",
    "corridor":   "#457B9D",
    "kitchen":    "#F4A261",
    "livingroom": "#FFBE0B",
    "stairs":     "#8338EC",
    "storeroom":  "#B07AA1",
    "dining":     "#76B7B2",
    "balcony":    "#2CA02C",
    "unknown":    "#AAAAAA",
}
print("Mappings loaded.")

Mappings loaded.


## 5. Paths

In [81]:
\
OBJECTS_DIR  = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects"
MODEL_PATH   = r"C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt"
DATASET_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B"
os.makedirs(DATASET_PATH, exist_ok=True)
print("Objects :", OBJECTS_DIR)
print("Model   :", MODEL_PATH)
print("Dataset :", DATASET_PATH)

Objects : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects
Model   : C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt
Dataset : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B


## 6. Load room OBJs and build cells

Each OBJ file contains one room type (multiple objects = multiple floor levels).  
`Topology.ByOBJPath(transposeAxes=True)` converts Rhino Y-up → Z-up.  
`Cell.ByFaces` builds a solid cell at progressively loose tolerances (HW02 approach).

Each OBJ file represents one room type. We import the geometry, extract the enclosed cell
volumes, and create **selector vertices** (internal points) carrying `room_type`, `label`,
`cell_color`, `zoning`, and `connectivity` dictionaries. These selectors are later used to
transfer labels onto the merged CellComplex cells.

In [82]:
ROOM_FILES = {
    "Bedroom":     ("bedroom",    os.path.join(OBJECTS_DIR, "Bedroom.obj")),
    "Living room": ("livingroom", os.path.join(OBJECTS_DIR, "Living room.obj")),
    "Kitchen":     ("kitchen",    os.path.join(OBJECTS_DIR, "Kitchen.obj")),
    "Corridor":    ("corridor",   os.path.join(OBJECTS_DIR, "Corridor.obj")),
    "Stair":       ("stairs",     os.path.join(OBJECTS_DIR, "Stair.obj")),
    "Bathroom":    ("bathroom",   os.path.join(OBJECTS_DIR, "Bathroom.obj")),
    "Balcony":     ("balcony",    os.path.join(OBJECTS_DIR, "balcony.obj")),
}

def build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None:
            return c
    return None

all_cells = []
selectors = []

for display_name, (room_type, obj_path) in ROOM_FILES.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    objs = Topology.ByOBJPath(obj_path, transposeAxes=True)
    if not isinstance(objs, list):
        objs = [objs] if objs else []

    label  = ROOM_LABEL[room_type]
    zoning = ZONING[room_type]
    conn   = NODE_CONNECTIVITY[room_type]
    color  = ROOM_COLOR[room_type]

    n_ok = 0
    for obj in objs:
        if obj is None:
            continue
        faces = Topology.Faces(obj) or []
        if len(faces) < 4:
            continue
        c = build_cell(faces)
        if c is None:
            continue

        d = Dictionary.ByKeysValues(
            ["room_type", "type", "label", "color", "cell_color",
             "feat_zoning_type_0", "feat_zoning_type_1",
             "feat_zoning_type_2", "feat_zoning_type_3",
             "feat_connectivity_0", "feat_connectivity_1",
             "feat_connectivity_2"],
            [room_type, display_name, label, color, color,
             zoning[0], zoning[1], zoning[2], zoning[3],
             conn[0], conn[1], conn[2]]
        )

        c  = Topology.SetDictionary(c, d)
        iv = Topology.InternalVertex(c)
        iv = Topology.SetDictionary(iv, d)
        selectors.append(iv)
        all_cells.append(c)
        n_ok += 1

    status = f"cells={n_ok}" if n_ok else "[SKIP] no cells built"
    print(f"  {display_name:20s} -> {room_type:12s}  label={label}  {status}")

print(f"\nTotal: {len(all_cells)} cells, {len(selectors)} selectors")

  Bedroom              -> bedroom       label=0  cells=5
  Living room          -> livingroom    label=1  cells=2
  Kitchen              -> kitchen       label=2  cells=1
  Corridor             -> corridor      label=4  cells=5
  Stair                -> stairs        label=5  cells=6
  Bathroom             -> bathroom      label=7  cells=6
  Balcony              -> balcony       label=8  cells=2

Total: 27 cells, 27 selectors


## 7. Build CellComplex and transfer room-type dictionaries

All room cells are merged into a single `CellComplex`. Then
`Topology.TransferDictionariesBySelectors` assigns the room-type dictionaries from the
selector vertices to the CellComplex cells.

In [83]:
cc = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
print("Topology type :", Topology.TypeAsString(cc))
print("Cells :", len(Topology.Cells(cc) or []))
print("Faces :", len(Topology.Faces(cc) or []))

# Progressive tolerance: start tight (avoids wrong assignments), widen only if needed.
# Balcony cells are spatially isolated so higher tolerance stays safe.
for _tol in [0.1, 0.5, 1.0, 2.0]:
    cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True, tolerance=_tol)
    _labelled = sum(1 for c in (Topology.Cells(cc) or [])
                    if Dictionary.ValueAtKey(Topology.Dictionary(c), "room_type"))
    print(f"  TransferDictionaries tolerance={_tol} → {_labelled}/{len(Topology.Cells(cc) or [])} cells labelled")
    if _labelled >= len(selectors):
        break

counts, unlabelled = Counter(), 0
for cell in (Topology.Cells(cc) or []):
    d  = Topology.Dictionary(cell)
    rt = Dictionary.ValueAtKey(d, "room_type")
    if rt:
        counts[rt] += 1
    else:
        unlabelled += 1

print("\nRoom distribution:")
for rt, n in sorted(counts.items(), key=lambda x: ROOM_LABEL.get(x[0], 99)):
    print(f"  {rt:15s}  label={ROOM_LABEL[rt]}  count={n}")
if unlabelled:
    print(f"  *** {unlabelled} unlabelled cell(s) (SelfMerge artifacts — expected)")
else:
    print("  All cells labelled.")

Topology type : CellComplex
Cells : 40
Faces : 249
  TransferDictionaries tolerance=0.1 → 27/40 cells labelled

Room distribution:
  bedroom          label=0  count=5
  livingroom       label=1  count=2
  kitchen          label=2  count=1
  corridor         label=4  count=5
  stairs           label=5  count=6
  bathroom         label=7  count=6
  balcony          label=8  count=2
  *** 13 unlabelled cell(s) (SelfMerge artifacts — expected)


## 8. Visualise CellComplex coloured by room type

In [84]:
# Build display copies: RemoveCoplanarFaces removes OBJ triangulation visually,
# then copy the color dictionary back (the cleaned topology is a new object).
# all_cells stays untouched — SelfMerge and AddApertures need the original geometry.
display_cells = []
for c in all_cells:
    c2 = Topology.RemoveCoplanarFaces(c, epsilon=0.1, tolerance=0.001, silent=True)
    if c2:
        c2 = Topology.RemoveCollinearEdges(c2) or c2
        c2 = Topology.SetDictionary(c2, Topology.Dictionary(c))
    display_cells.append(c2 if c2 else c)

Topology.Show(
    display_cells,
    selectors,
    faceColorKey="color",
    faceOpacity=0.4,
    showEdges=True, edgeWidth=3,
    showVertices=True, vertexSize=10,
    vertexLabelKey="type",
    showVertexLabel=True,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

Face.ByWire - Warning: Could not create face by external boundary. Trying cleaned wire.
Face.ByWire - Warning: Could not build a single face from the input wire parameter. Returning a list of faces.
Face.Area - Warning: The input face parameter is not a valid topologic face. Returning None.
Face.ByWires - Error: The operation failed. Returning None.
caller name: RemoveCollinearEdges
Face.ByWire - Warning: Could not create face by external boundary. Trying cleaned wire.
Face.ByWire - Warning: Could not build a single face from the input wire parameter. Returning a list of faces.
Face.Area - Warning: The input face parameter is not a valid topologic face. Returning None.
Face.ByWires - Error: The operation failed. Returning None.
caller name: RemoveCollinearEdges
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Face.ByWire - Warning: Could not create face by external bounda

## 9. Load door OBJs as apertures

Doors are modelled as planar face objects. We collect faces from three door files:
- `doors2.obj` — standard interior doors (type `door`)
- `Entrance door.obj` — exterior / entrance doors (type `entrance_door`)
- `Passage Door.objbak` — open passage connections (type `passage`)

Each face is tagged with a `door_type` dictionary, then passed as apertures to
`Topology.AddApertures`. When `Graph.ByTopology(cc, directApertures=True)` is called,
it creates graph edges only between cells whose shared face carries an aperture.

In [85]:
DOOR_FILES = {
    "door":          os.path.join(OBJECTS_DIR, "doors2.obj"),
    "entrance_door": os.path.join(OBJECTS_DIR, "Entrance door.obj"),
    "passage":       os.path.join(OBJECTS_DIR, "Passage Door.objbak"),
}

apertures = []

for door_type, obj_path in DOOR_FILES.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    conn = DOOR_CONNECTIVITY[door_type]
    objs = Topology.ByOBJPath(obj_path, transposeAxes=True)
    if not isinstance(objs, list):
        objs = [objs] if objs else []

    faces_for_type = []
    for obj in objs:
        if obj is None:
            continue
        faces = Topology.Faces(obj) or []
        if faces:
            faces_for_type.extend(faces)
        else:
            for w in (Topology.Wires(obj) or []):
                f = Face.ByWire(w)
                if f is None:
                    w2 = Topology.RemoveCollinearEdges(w)
                    f  = Face.ByWire(w2) if w2 else None
                if f is not None:
                    faces_for_type.append(f)

    for f in faces_for_type:
        d = Dictionary.ByKeysValues(
            ["door_type",
             "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"],
            [door_type, conn[0], conn[1], conn[2]]
        )
        f = Topology.SetDictionary(f, d)
        apertures.append(f)

    print(f"  {door_type:20s} -> {len(faces_for_type)} aperture faces")

print(f"\nTotal apertures: {len(apertures)}")

  door                 -> 11 aperture faces
  entrance_door        -> 4 aperture faces
  passage              -> 9 aperture faces

Total apertures: 24


## 10. Add apertures to CellComplex

In [86]:
if not apertures:
    print("No apertures found — graph will use direct cell adjacency.")
else:
    best_cc      = cc
    best_matched = 0

    for tol in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0]:
        cc_try  = Topology.AddApertures(
            cc, apertures, exclusive=False, subTopologyType="Face", tolerance=tol)
        matched = sum(
            1 for f in (Topology.Faces(cc_try) or [])
            if Topology.Apertures(f)
        )
        print(f"  tolerance={tol:5.3f}  ->  {matched}/{len(apertures)} faces matched")
        if matched > best_matched:
            best_matched = matched
            best_cc      = cc_try
        if matched == len(apertures):
            break

    cc = best_cc

    # Diagnose unmatched apertures: compare each aperture centroid to nearest CC face centroid
    if best_matched < len(apertures):
        print("\n--- Unmatched aperture diagnosis ---")
        cc_faces = Topology.Faces(cc) or []
        cc_face_centroids = []
        for f in cc_faces:
            vs = Topology.Vertices(f) or []
            if vs:
                cx = sum(Vertex.X(v) for v in vs) / len(vs)
                cy = sum(Vertex.Y(v) for v in vs) / len(vs)
                cz = sum(Vertex.Z(v) for v in vs) / len(vs)
                cc_face_centroids.append((cx, cy, cz))

        # Find matched aperture centroids (faces that have apertures)
        matched_centroids = set()
        for f in cc_faces:
            aps = Topology.Apertures(f) or []
            for ap in aps:
                if ap is None:
                    continue
                vs = Topology.Vertices(ap) or []
                if vs:
                    cx = round(sum(Vertex.X(v) for v in vs) / len(vs), 2)
                    cy = round(sum(Vertex.Y(v) for v in vs) / len(vs), 2)
                    cz = round(sum(Vertex.Z(v) for v in vs) / len(vs), 2)
                    matched_centroids.add((cx, cy, cz))

        def centroid(ap_face):
            vs = Topology.Vertices(ap_face) or []
            if not vs:
                return None
            return (
                sum(Vertex.X(v) for v in vs) / len(vs),
                sum(Vertex.Y(v) for v in vs) / len(vs),
                sum(Vertex.Z(v) for v in vs) / len(vs),
            )

        def nearest_cc_face_dist(pos):
            best = float("inf")
            for fc in cc_face_centroids:
                d = ((pos[0]-fc[0])**2 + (pos[1]-fc[1])**2 + (pos[2]-fc[2])**2)**0.5
                if d < best:
                    best = d
            return best

        for ap in apertures:
            c = centroid(ap)
            if c is None:
                continue
            cr = (round(c[0], 2), round(c[1], 2), round(c[2], 2))
            if cr not in matched_centroids:
                dist = nearest_cc_face_dist(c)
                dt   = Dictionary.ValueAtKey(Topology.Dictionary(ap), "door_type") or "?"
                print(f"  UNMATCHED  door_type={dt}  pos=({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f})  "
                      f"nearest_face_dist={dist:.4f}")

    faces_with_ap = best_matched
    unmatched     = len(apertures) - faces_with_ap
    print(f"\nFaces carrying apertures : {faces_with_ap}")
    if unmatched:
        print(f"Unmatched apertures      : {unmatched}")

  tolerance=0.001  ->  23/24 faces matched
  tolerance=0.010  ->  23/24 faces matched
  tolerance=0.050  ->  23/24 faces matched
  tolerance=0.100  ->  23/24 faces matched
  tolerance=0.500  ->  23/24 faces matched
  tolerance=1.000  ->  27/24 faces matched
  tolerance=2.000  ->  33/24 faces matched

Faces carrying apertures : 33
Unmatched apertures      : -9


## 11. Build the room adjacency graph

`Graph.ByTopology` with `directApertures=True` creates:
- One **vertex** per cell (room)  
- One **edge** between two cells whose shared face carries a door aperture

This exactly mirrors the MSD dataset construction.

In [87]:
graph = Graph.ByTopology(cc, direct=False, directApertures=True)

n_raw = len(Graph.Edges(graph) or [])
print(f"Graph (raw): {len(Graph.Vertices(graph) or [])} vertices, {n_raw} edges")

if n_raw == 0:
    print("No aperture edges — falling back to direct cell adjacency.")
    graph = Graph.ByTopology(cc, direct=True, directApertures=False)

# --- Same-level filter (mirrors 02's same_level z_tol=0.60) ---
SAME_FLOOR_Z_TOL = 1.5

all_verts_raw = Graph.Vertices(graph) or []
all_edges_raw = Graph.Edges(graph) or []

def _get_rt(v):
    return Dictionary.ValueAtKey(Topology.Dictionary(v), "room_type")

valid_edges = []
removed_log = []
for e in all_edges_raw:
    sv, ev = Edge.StartVertex(e), Edge.EndVertex(e)
    sv_rt, ev_rt = _get_rt(sv), _get_rt(ev)
    if sv_rt == "stairs" or ev_rt == "stairs":
        valid_edges.append(e)
        continue
    dz = abs(Vertex.Z(sv) - Vertex.Z(ev))
    if dz <= SAME_FLOOR_Z_TOL:
        valid_edges.append(e)
    else:
        removed_log.append(
            f"  removed: {sv_rt}(Z={Vertex.Z(sv):.2f}) ↔ {ev_rt}(Z={Vertex.Z(ev):.2f})  ΔZ={dz:.2f}"
        )

if removed_log:
    for msg in removed_log:
        print(msg)
    graph = Graph.ByVerticesEdges(all_verts_raw, valid_edges)
    print(f"Removed {len(removed_log)} cross-floor edge(s)")

n_v = len(Graph.Vertices(graph) or [])
n_e = len(Graph.Edges(graph) or [])
print(f"Graph (filtered): {n_v} vertices, {n_e} edges")

# --- Isolation patch ---
VALID_NEIGHBOURS = {
    "bedroom":    {"corridor", "bathroom", "livingroom", "storeroom", "balcony"},
    "bathroom":   {"bedroom", "corridor"},
    "corridor":   {"bedroom", "bathroom", "livingroom", "kitchen", "stairs",
                   "storeroom", "dining"},
    "stairs":     {"corridor"},
    "livingroom": {"corridor", "kitchen", "dining", "bedroom"},
    "kitchen":    {"corridor", "livingroom", "dining"},
    "dining":     {"corridor", "livingroom", "kitchen"},
    "storeroom":  {"corridor", "bedroom"},
    "balcony":    {"bedroom", "livingroom"},
}

def _dist3sq(a, b):
    return sum((x-y)**2 for x, y in zip(a, b))

def _face_cent_key(face, prec=1):
    vs = Topology.Vertices(face) or []
    if not vs: return None
    return (round(sum(Vertex.X(v) for v in vs)/len(vs), prec),
            round(sum(Vertex.Y(v) for v in vs)/len(vs), prec),
            round(sum(Vertex.Z(v) for v in vs)/len(vs), prec))

labelled_verts = [v for v in (Graph.Vertices(graph) or []) if _get_rt(v) is not None]
cc_cells = Topology.Cells(cc) or []

iv_to_cell, cell_faces_map, cell_z_map = {}, {}, {}
for ci, c in enumerate(cc_cells):
    iv = Topology.InternalVertex(c)
    if iv:
        k = tuple(round(x, 2) for x in Vertex.Coordinates(iv))
        iv_to_cell[k] = (ci, c)
        cell_z_map[ci] = Vertex.Z(iv)
    fks = frozenset(k for f in (Topology.Faces(c) or []) for k in [_face_cent_key(f)] if k)
    cell_faces_map[ci] = fks

def _find_cc_cell(v):
    vk = tuple(round(x, 2) for x in Vertex.Coordinates(v))
    if vk in iv_to_cell: return iv_to_cell[vk]
    best_ci, best_c, bd = None, None, float("inf")
    for ci, c in enumerate(cc_cells):
        iv = Topology.InternalVertex(c)
        if iv:
            d = _dist3sq(Vertex.Coordinates(v), Vertex.Coordinates(iv))
            if d < bd: bd, best_ci, best_c = d, ci, c
    return (best_ci, best_c)

def _adj_cells(ci):
    own = cell_faces_map[ci]
    return [cj for cj, fks in cell_faces_map.items() if cj != ci and own & fks]

isolated = [v for v in labelled_verts if not (Graph.AdjacentVertices(graph, v) or [])]
if isolated:
    print(f"\n{len(isolated)} isolated room(s) — patching:")
    vpos_to_gv = {tuple(round(x, 2) for x in Vertex.Coordinates(v)): v for v in labelled_verts}
    for iso_v in isolated:
        iso_rt = _get_rt(iso_v)
        iso_z  = Vertex.Z(iso_v)
        valid  = VALID_NEIGHBOURS.get(iso_rt, set())
        iso_ci, iso_c = _find_cc_cell(iso_v)
        if iso_c is None: print(f"  {iso_rt}: no CC cell"); continue
        patched = 0
        for adj_ci in _adj_cells(iso_ci):
            adj_c  = cc_cells[adj_ci]
            adj_rt = Dictionary.ValueAtKey(Topology.Dictionary(adj_c), "room_type")
            if adj_rt not in valid: continue
            adj_z  = cell_z_map.get(adj_ci, iso_z + 99)
            if abs(adj_z - iso_z) > SAME_FLOOR_Z_TOL: continue
            adj_iv = Topology.InternalVertex(adj_c)
            if not adj_iv: continue
            adj_k  = tuple(round(x, 2) for x in Vertex.Coordinates(adj_iv))
            adj_gv = vpos_to_gv.get(adj_k) or min(
                labelled_verts, key=lambda v: _dist3sq(Vertex.Coordinates(v), Vertex.Coordinates(adj_iv)))
            if adj_gv is iso_v: continue
            e_new = Edge.ByVertices([iso_v, adj_gv])
            if e_new:
                conn = DOOR_CONNECTIVITY["door"]
                e_new = Topology.SetDictionary(e_new, Dictionary.ByKeysValues(
                    ["door_type","feat_connectivity_0","feat_connectivity_1","feat_connectivity_2"],
                    ["door", conn[0], conn[1], conn[2]]))
                try:
                    graph = Graph.AddEdge(graph, e_new, tolerance=0.5)
                    patched += 1
                    print(f"  patched: {iso_rt} ↔ {adj_rt} (ΔZ={abs(adj_z-iso_z):.2f})")
                except Exception as ex:
                    print(f"  AddEdge error: {ex}")
        if patched == 0:
            print(f"  {iso_rt} at Z={iso_z:.2f}: no same-floor valid neighbour (fix door in Rhino)")
else:
    print("All labelled rooms connected.")

# --- Front balcony ↔ entrance stair: explicit connection ---
# Mirrors 02's cell-27 rule 3. Only the front balcony (spatially nearest to the entrance
# stair) gets this edge — the back balcony keeps only its aperture-based connections.
stair_sel_idxs_cg   = [i for i, s in enumerate(selectors)
                        if Dictionary.ValueAtKey(Topology.Dictionary(s), "room_type") == "stairs"]
entrance_sel_set_cg = set(stair_sel_idxs_cg[:2])   # first 2 = entrance stair (objects 1-2)

all_g_verts_cg      = Graph.Vertices(graph) or []
balcony_gvs_cg      = [v for v in all_g_verts_cg if _get_rt(v) == "balcony"]
stair_gvs_cg        = [v for v in all_g_verts_cg if _get_rt(v) == "stairs"]

def _nearest_sel_cg(v):
    p = Vertex.Coordinates(v)
    return min(range(len(selectors)),
               key=lambda i: _dist3sq(p, Vertex.Coordinates(selectors[i])))

entrance_stair_gvs_cg = [sv for sv in stair_gvs_cg
                          if _nearest_sel_cg(sv) in entrance_sel_set_cg]

print(f"\nBalcony graph vertices    : {len(balcony_gvs_cg)}")
print(f"Entrance stair graph verts: {len(entrance_stair_gvs_cg)}")

added_bs = 0
if balcony_gvs_cg and entrance_stair_gvs_cg:
    # Only connect the ONE balcony that is spatially closest to the entrance stair.
    # This is the front/entrance balcony; the back balcony stays unconnected to stairs.
    front_balcony = min(
        balcony_gvs_cg,
        key=lambda bv: min(
            _dist3sq(Vertex.Coordinates(bv), Vertex.Coordinates(esv))
            for esv in entrance_stair_gvs_cg
        )
    )
    front_z = Vertex.Z(front_balcony)
    print(f"Front balcony Z={front_z:.2f}  "
          f"(back balcony excluded from entrance-stair connection)")

    for esv in entrance_stair_gvs_cg:
        already = Graph.AdjacentVertices(graph, front_balcony) or []
        if any(_dist3sq(Vertex.Coordinates(esv), Vertex.Coordinates(adj)) < 0.01
               for adj in already):
            continue
        e_new = Edge.ByVertices([front_balcony, esv])
        if e_new:
            conn  = DOOR_CONNECTIVITY["entrance_door"]
            e_new = Topology.SetDictionary(e_new, Dictionary.ByKeysValues(
                ["door_type","feat_connectivity_0","feat_connectivity_1","feat_connectivity_2"],
                ["entrance_door", conn[0], conn[1], conn[2]]))
            try:
                graph = Graph.AddEdge(graph, e_new, tolerance=0.5)
                added_bs += 1
                print(f"  Added: front_balcony ↔ entrance_stair (entrance_door)")
            except Exception as ex:
                print(f"  balcony-stair AddEdge: {ex}")
elif balcony_gvs_cg and not entrance_stair_gvs_cg:
    print("  WARNING: no entrance stair graph vertices — check Stair.obj load order")

print(f"\nFinal: {len(Graph.Edges(graph) or [])} edges  (+{added_bs} front-balcony-stair)")

Graph (raw): 40 vertices, 27 edges
  removed: bathroom(Z=3.89) ↔ bathroom(Z=1.04)  ΔZ=2.86
  removed: corridor(Z=3.89) ↔ bathroom(Z=1.04)  ΔZ=2.86
  removed: kitchen(Z=3.89) ↔ None(Z=2.08)  ΔZ=1.82
  removed: livingroom(Z=3.89) ↔ livingroom(Z=1.04)  ΔZ=2.86
Removed 4 cross-floor edge(s)
Graph (filtered): 40 vertices, 23 edges

2 isolated room(s) — patching:
  patched: bathroom ↔ corridor (ΔZ=0.00)
  patched: balcony ↔ livingroom (ΔZ=0.00)

Balcony graph vertices    : 2
Entrance stair graph verts: 2
Front balcony Z=1.04  (back balcony excluded from entrance-stair connection)
  Added: front_balcony ↔ entrance_stair (entrance_door)
  Added: front_balcony ↔ entrance_stair (entrance_door)

Final: 27 edges  (+2 front-balcony-stair)


## 11B. Visualise the room adjacency graph

In [88]:
DOOR_COLOR = {
    "passage":       "#59A14F",
    "door":          "#4E79A7",
    "entrance_door": "#F28E2B",
}

# Area per cell — selectors and all_cells are built 1-to-1
cell_areas = [Cell.SurfaceArea(c) for c in all_cells]
min_area   = min(cell_areas)
max_area   = max(cell_areas)

def area_to_size(area):
    return 12 + int(48 * (area - min_area) / (max_area - min_area + 1))

# Selector positions for nearest-neighbour lookup
sel_pos = [(Vertex.X(s), Vertex.Y(s), Vertex.Z(s)) for s in selectors]

def _d3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2) ** 0.5

def nearest_sel_idx(v):
    p = (Vertex.X(v), Vertex.Y(v), Vertex.Z(v))
    return min(range(len(sel_pos)), key=lambda i: _d3(p, sel_pos[i]))

def nearest_area(v):
    return cell_areas[nearest_sel_idx(v)]

def get_rt(v):
    return Dictionary.ValueAtKey(Topology.Dictionary(v), "room_type")

def is_stair_v(v):
    return get_rt(v) == "stairs"

def is_room_v(v):
    return get_rt(v) is not None

all_verts = Graph.Vertices(graph) or []
all_edges = Graph.Edges(graph) or []

room_verts      = [v for v in all_verts if is_room_v(v)]
stair_verts     = [v for v in room_verts if is_stair_v(v)]
non_stair_verts = [v for v in room_verts if not is_stair_v(v)]

# Split stair selectors into entrance (first 2 in load order = Stair.obj objects 1-2)
# and main shaft (last 4 = objects 3-6), matching the Stair.obj object ordering.
stair_sel_indices = [i for i, s in enumerate(selectors)
                     if Dictionary.ValueAtKey(Topology.Dictionary(s), "room_type") == "stairs"]
entrance_sel_set  = set(stair_sel_indices[:2])   # objects 1-2: entrance stair
main_sel_set      = set(stair_sel_indices[2:])    # objects 3-6: main 4-floor shaft

stair_main_verts = [v for v in stair_verts if nearest_sel_idx(v) in main_sel_set]
stair_ext_verts  = [v for v in stair_verts if nearest_sel_idx(v) in entrance_sel_set]

def _make_stair_rep(verts, label):
    if not verts:
        return None
    sx = sum(Vertex.X(v) for v in verts) / len(verts)
    sy = sum(Vertex.Y(v) for v in verts) / len(verts)
    sz = sum(Vertex.Z(v) for v in verts) / len(verts)
    area = sum(nearest_area(v) for v in verts)
    rep  = Vertex.ByCoordinates(sx, sy, sz)
    rep  = Topology.SetDictionary(rep, Dictionary.ByKeysValues(
        ["v_color", "v_label", "v_size", "room_type"],
        [ROOM_COLOR["stairs"], label, area_to_size(area), "stairs"]
    ))
    return rep

stair_main_rep = _make_stair_rep(stair_main_verts, "stairs")
stair_ext_rep  = _make_stair_rep(stair_ext_verts,  "stairs")

# Style non-stair vertices with area-scaled sizes
vis_verts = []
for v in non_stair_verts:
    rt   = get_rt(v) or "unknown"
    area = nearest_area(v)
    d    = Dictionary.SetValuesAtKeys(
        Topology.Dictionary(v),
        ["v_color", "v_label", "v_size"],
        [ROOM_COLOR.get(rt, "#AAAAAA"), rt, area_to_size(area)]
    )
    vis_verts.append(Topology.SetDictionary(v, d))

if stair_main_rep:
    vis_verts.append(stair_main_rep)
if stair_ext_rep:
    vis_verts.append(stair_ext_rep)

# Build a lookup: for any stair graph vertex, which rep node does it map to?
def stair_rep_for(v):
    if not is_stair_v(v):
        return v
    return stair_main_rep if nearest_sel_idx(v) in main_sel_set else stair_ext_rep

# Edges: remap stair endpoints to their rep node
vis_edges  = []
seen_pairs = set()
for e in all_edges:
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    if not is_room_v(sv) or not is_room_v(ev):
        continue
    sv2 = stair_rep_for(sv)
    ev2 = stair_rep_for(ev)
    if sv2 is ev2:
        continue
    key = tuple(sorted([id(sv2), id(ev2)]))
    if key in seen_pairs:
        continue
    seen_pairs.add(key)
    new_e = Edge.ByVertices([sv2, ev2])
    if new_e:
        dt    = Dictionary.ValueAtKey(Topology.Dictionary(e), "door_type") or "door"
        new_e = Topology.SetDictionary(new_e, Dictionary.ByKeysValues(
            ["e_color"], [DOOR_COLOR.get(dt, "#888888")]
        ))
        vis_edges.append(new_e)

print(f"Main stair     : {len(stair_main_verts)} cells → 1 node")
print(f"Entrance stair : {len(stair_ext_verts)} cells → 1 node")
print(f"Vis nodes : {len(vis_verts)}  |  Vis edges : {len(vis_edges)}")

Topology.Show(
    vis_verts + vis_edges,
    vertexSizeKey="v_size",
    vertexColorKey="v_color",
    showVertexLabel=True, vertexLabelKey="v_label", vertexLabelFontSize=14,
    edgeWidth=3, edgeColorKey="e_color",
    backgroundColor="white",
    width=900, height=700,
    renderer=renderer
)

Main stair     : 4 cells → 1 node
Entrance stair : 2 cells → 1 node
Vis nodes : 23  |  Vis edges : 24


## 12. Verify graph vertex and edge dictionaries

In [89]:
vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph) or []

print("Sample vertex dictionaries (first 6):")
for v in vertices[:6]:
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type")
    lb = Dictionary.ValueAtKey(d, "label")
    z0 = Dictionary.ValueAtKey(d, "feat_zoning_type_0")
    c0 = Dictionary.ValueAtKey(d, "feat_connectivity_0")
    print(f"  room_type={str(rt):12s}  label={lb}  zoning[0]={z0}  conn[0]={c0}")

print(f"\nSample edge dictionaries (first 5):")
for e in edges[:5]:
    d  = Topology.Dictionary(e)
    dt = Dictionary.ValueAtKey(d, "door_type")
    ks = Dictionary.Keys(d)
    print(f"  door_type={dt}  keys={ks}")

Sample vertex dictionaries (first 6):
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=corridor      label=4  zoning[0]=0  conn[0]=1
  room_type=bathroom      label=7  zoning[0]=0  conn[0]=0
  room_type=corridor      label=4  zoning[0]=0  conn[0]=1
  room_type=None          label=None  zoning[0]=None  conn[0]=None
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0

Sample edge dictionaries (first 5):
  door_type=entrance_door  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'relationship', 'src', 'type']
  door_type=passage  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'relationship', 'src', 'type']
  door_type=passage  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'relationship'

## 13. Export CSVs in MSD schema

| File | Columns |
|---|---|
| `graphs.csv` | `graph_id`, `num_nodes` |
| `nodes.csv` | `graph_id`, `node_id`, `label`, features, masks |
| `edges.csv` | `graph_id`, `src_id`, `dst_id`, `feat_connectivity_0..2` |

In [90]:
def vkey(v, tol=3):
    return tuple(round(x, tol) for x in Vertex.Coordinates(v))

def gv(d, k, default=0):
    val = Dictionary.ValueAtKey(d, k)
    return val if val is not None else default

# Drop the 13 SelfMerge artifact cells — keep only labelled room cells
labelled = []
for i, v in enumerate(vertices):
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type")
    if rt is not None:
        labelled.append((i, v))

print(f"Labelled vertices : {len(labelled)} / {len(vertices)}")

old_to_new   = {old_i: new_i for new_i, (old_i, _) in enumerate(labelled)}
coord_to_new = {vkey(v): old_to_new[old_i] for new_i, (old_i, v) in enumerate(labelled)}

# nodes.csv
nodes_rows = []
for new_i, (old_i, v) in enumerate(labelled):
    d = Topology.Dictionary(v)
    nodes_rows.append({
        "graph_id": 0, "node_id": new_i,
        "label":               int(gv(d, "label", 0)),
        "feat_zoning_type_0":  int(gv(d, "feat_zoning_type_0")),
        "feat_zoning_type_1":  int(gv(d, "feat_zoning_type_1")),
        "feat_zoning_type_2":  int(gv(d, "feat_zoning_type_2")),
        "feat_zoning_type_3":  int(gv(d, "feat_zoning_type_3")),
        "feat_connectivity_0": int(gv(d, "feat_connectivity_0")),
        "feat_connectivity_1": int(gv(d, "feat_connectivity_1", 1)),
        "feat_connectivity_2": int(gv(d, "feat_connectivity_2")),
        "train_mask": 0, "val_mask": 0, "test_mask": 1,
    })

# edges.csv — bidirectional (A→B and B→A) to match nx.Graph undirected training format
# only include edges where both endpoints are labelled rooms
edges_rows = []
for e in edges:
    sk  = vkey(Edge.StartVertex(e))
    ek  = vkey(Edge.EndVertex(e))
    src = coord_to_new.get(sk)
    dst = coord_to_new.get(ek)
    if src is None or dst is None:
        continue
    d = Topology.Dictionary(e)
    feat = {
        "feat_connectivity_0": int(gv(d, "feat_connectivity_0")),
        "feat_connectivity_1": int(gv(d, "feat_connectivity_1", 1)),
        "feat_connectivity_2": int(gv(d, "feat_connectivity_2")),
    }
    edges_rows.append({"graph_id": 0, "src_id": src, "dst_id": dst, **feat})
    edges_rows.append({"graph_id": 0, "src_id": dst, "dst_id": src, **feat})

pd.DataFrame([{"graph_id": 0, "num_nodes": len(nodes_rows)}]).to_csv(
    os.path.join(DATASET_PATH, "graphs.csv"), index=False)
pd.DataFrame(nodes_rows).to_csv(
    os.path.join(DATASET_PATH, "nodes.csv"), index=False)
pd.DataFrame(edges_rows).to_csv(
    os.path.join(DATASET_PATH, "edges.csv"), index=False)

print(f"graphs.csv : 1 graph")
print(f"nodes.csv  : {len(nodes_rows)} nodes (labelled rooms only)")
print(f"edges.csv  : {len(edges_rows)} rows ({len(edges_rows)//2} doors × 2 directions)")
print(f"Saved to   : {DATASET_PATH}")

Labelled vertices : 27 / 40
graphs.csv : 1 graph
nodes.csv  : 27 nodes (labelled rooms only)
edges.csv  : 52 rows (26 doors × 2 directions)
Saved to   : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B


## 14. Inspect exported CSVs

In [91]:
nodes_df = pd.read_csv(os.path.join(DATASET_PATH, "nodes.csv"))
edges_df = pd.read_csv(os.path.join(DATASET_PATH, "edges.csv"))
print("nodes.csv")
print(nodes_df.to_string(index=False))
print()
print("edges.csv")
print(edges_df.to_string(index=False))

nodes.csv
 graph_id  node_id  label  feat_zoning_type_0  feat_zoning_type_1  feat_zoning_type_2  feat_zoning_type_3  feat_connectivity_0  feat_connectivity_1  feat_connectivity_2  train_mask  val_mask  test_mask
        0        0      0                   1                   0                   0                   0                    0                    1                    0           0         0          1
        0        1      4                   0                   1                   0                   0                    1                    0                    0           0         0          1
        0        2      7                   0                   0                   1                   0                    0                    1                    0           0         0          1
        0        3      4                   0                   1                   0                   0                    1                    0                    0           0  

## 15. Load dataset into PyG

In [92]:
pyg = PyG.ByCSVPath(
    path=DATASET_PATH,
    level="node",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical",
)
print(pyg)

## 16. Load the pretrained MSD node classifier

In [93]:
pyg.LoadModel(MODEL_PATH)
print("Model loaded.")

Model loaded.


## 17. Predict room types

In [94]:
def to_class(val):
    a = np.squeeze(np.asarray(val))
    if a.ndim == 0: return int(a)
    if a.ndim == 1: return int(np.argmax(a)) if a.size > 1 else int(a[0])
    raise ValueError(f"Unexpected shape {a.shape}")

report    = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
pred_list = report["pred"]
true_list = report["y_true"]
prob_list = report.get("prob", None)

LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}

rows = []
for g_idx, data in enumerate(pyg.data_list):
    gid   = int(data.graph_id.item()) if hasattr(data, "graph_id") else g_idx
    n     = data.num_nodes
    gp    = np.asarray(pred_list[g_idx])
    gt    = np.asarray(true_list[g_idx])
    gprob = np.asarray(prob_list[g_idx]) if prob_list else None
    for ni in range(n):
        yt  = to_class(gt[ni])
        yp  = to_class(gp[ni])
        row = {"graph_id": gid, "node_id": ni, "y_true": yt, "y_pred": yp}
        if gprob is not None:
            p = np.squeeze(np.asarray(gprob[ni]))
            if p.ndim == 1 and yp < p.size:
                row["y_pred_prob"] = float(p[yp])
        rows.append(row)

pred_df = pd.DataFrame(rows)
pred_df["true_name"] = pred_df["y_true"].map(LABEL_NAME)
pred_df["pred_name"] = pred_df["y_pred"].map(LABEL_NAME)

PRED_CSV = os.path.join(DATASET_PATH, "node_predictions.csv")
pred_df.to_csv(PRED_CSV, index=False)

correct = (pred_df["y_true"] == pred_df["y_pred"]).sum()
total   = len(pred_df)
print(f"Predictions: {correct}/{total} correct = {correct/total:.1%}")
print(f"Saved: {PRED_CSV}")
print(pred_df[["node_id","true_name","pred_name"]].to_string(index=False))

Predictions: 17/27 correct = 63.0%
Saved: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B\node_predictions.csv
 node_id  true_name pred_name
       0    bedroom   bedroom
       1   corridor   kitchen
       2   bathroom  bathroom
       3   corridor   kitchen
       4    bedroom   bedroom
       5   bathroom  bathroom
       6     stairs    stairs
       7    bedroom   bedroom
       8   bathroom  bathroom
       9     stairs    stairs
      10   corridor   kitchen
      11    bedroom   bedroom
      12   bathroom  bathroom
      13   bathroom  bathroom
      14   corridor  corridor
      15 livingroom   kitchen
      16     stairs  bathroom
      17     stairs storeroom
      18   corridor   kitchen
      19   bathroom  bathroom
      20    kitchen   kitchen
      21 livingroom  corridor
      22     stairs  bathroom
      23     stairs storeroom
      24    bedroom   bedroom
      25    balcony   balcony
      26    balcony   balcony


## 18. Visualise true vs predicted labels

Misclassified nodes shown in **red** (size 30).  
Correctly classified nodes use a colour scale.

In [95]:
LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}

pred_lookup  = pred_df.set_index("node_id")[["y_true", "y_pred"]].to_dict("index")

# Style only the 23 labelled vertices — artifact cells are never added to vis_verts
vis_verts   = []
new_i_to_v  = {}
for new_i, (old_i, v) in enumerate(labelled):
    row = pred_lookup.get(new_i, {"y_true": 0, "y_pred": 0})
    yt  = int(row["y_true"])
    yp  = int(row["y_pred"])
    d   = Topology.Dictionary(v)
    if yt != yp:
        sz = 30
        tc = pc = "red"
    else:
        sz = 14
        tc = Color.ByValueInRange(yt, minValue=0, maxValue=8)
        pc = Color.ByValueInRange(yp, minValue=0, maxValue=8)
    d = Dictionary.SetValuesAtKeys(
        d,
        ["true_color", "pred_color", "node_size", "true_label", "pred_label"],
        [tc, pc, sz, LABEL_NAME.get(yt, str(yt)), LABEL_NAME.get(yp, str(yp))]
    )
    v = Topology.SetDictionary(v, d)
    vis_verts.append(v)
    new_i_to_v[new_i] = v

# Build edges directly between labelled vertices (skips artifact cells entirely)
vis_edges = []
seen_epairs = set()
for e in edges:
    sk  = vkey(Edge.StartVertex(e))
    ek  = vkey(Edge.EndVertex(e))
    src = coord_to_new.get(sk)
    dst = coord_to_new.get(ek)
    if src is None or dst is None:
        continue
    key = tuple(sorted([src, dst]))
    if key in seen_epairs:
        continue
    seen_epairs.add(key)
    new_e = Edge.ByVertices([new_i_to_v[src], new_i_to_v[dst]])
    if new_e:
        vis_edges.append(new_e)

correct = sum(1 for r in pred_lookup.values() if r["y_true"] == r["y_pred"])
print(f"Correct: {correct}/{len(vis_verts)}  Isolated node 13 (bathroom) has no edges — expected.")

print("--- True labels ---")
Topology.Show(
    vis_verts + vis_edges,
    vertexSize=6, vertexSizeKey="node_size",
    vertexColorKey="true_color",
    showVertexLabel=True, vertexLabelKey="true_label", vertexLabelFontSize=14,
    edgeWidth=2,
    backgroundColor="white",
    width=900, height=600, renderer=renderer
)

print("--- Predicted labels ---")
Topology.Show(
    vis_verts + vis_edges,
    vertexSize=6, vertexSizeKey="node_size",
    vertexColorKey="pred_color",
    showVertexLabel=True, vertexLabelKey="pred_label", vertexLabelFontSize=14,
    edgeWidth=2,
    backgroundColor="white",
    width=900, height=600, renderer=renderer
)

Correct: 17/27  Isolated node 13 (bathroom) has no edges — expected.
--- True labels ---


--- Predicted labels ---
